In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

In [ ]:
# Load Data
df = pd.read_csv('../arthur_hf_scraping/bnp/bnp_funds_returns.csv', index_col=0, parse_dates=True)

# Train/Test Split
train_start = '2020-01-01'
train_end = '2022-12-31'
test_start = '2023-01-01'
test_end = '2026-12-31'

print(f"- Number of funds in the dataset originally: {len(df.columns)}")
print(f"- Number of funds that have a full history during train and test: {df.loc[train_start:test_end].dropna(axis=1, how='any').shape[1]}")

df_train = df.loc[train_start:train_end].dropna(axis=1, how='any')
df_test = df.loc[test_start:test_end]


In [ ]:
# Selection Criteria
vol = df_train.std() * np.sqrt(12)
ret = df_train.mean()

# Top 35% of volatility (quantile 0.65 means we want the highest 35%)
vol_threshold = vol.quantile(0.65)
high_vol_funds = vol[vol >= vol_threshold].index

# Filter returns for high vol funds
ret_filtered = ret.loc[high_vol_funds]

# Top 50% of returns among high vol funds
ret_threshold = ret_filtered.quantile(0.50)
final_funds = ret_filtered[ret_filtered >= ret_threshold].index.tolist()

print(f"- Number of funds selected as our 'top funds': {len(final_funds)}")

# Filter test set
test_data = df_test[final_funds].dropna(axis=1, how='any')
print(f"Test data shape: {test_data.shape}")


In [ ]:
# Monte Carlo Heatmaps for N in [10, 15, 20]
def calculate_mdd(returns):
    cum_ret = (1 + returns).cumprod()
    running_max = cum_ret.cummax()
    drawdown = (cum_ret - running_max) / running_max
    return drawdown.min()

N_heatmap_values = [10, 15, 20]
n_simulations = 5000

np.random.seed(42)
available_funds = test_data.columns.tolist()

# Dictionaries to store the grids for the final overlay plot
vol_grids = {}
mdd_grids = {}

for N in N_heatmap_values:
    if len(available_funds) < N:
        print(f"Not enough funds to simulate N={N}.")
        continue
        
    sim_returns = []
    sim_vols = []
    sim_mdds = []
    
    for _ in tqdm(range(n_simulations), desc=f'Simulating Portfolios (N={N})'):
        selected_funds = np.random.choice(available_funds, size=N, replace=False)
        portfolio_returns = test_data[selected_funds].mean(axis=1) # Equal Weight
        
        ann_ret = portfolio_returns.mean() * 12
        ann_vol = portfolio_returns.std() * np.sqrt(12)
        mdd = calculate_mdd(portfolio_returns)
        
        sim_returns.append(ann_ret)
        sim_vols.append(ann_vol)
        sim_mdds.append(mdd)

    sim_returns_arr = np.array(sim_returns)
    sim_vols_arr = np.array(sim_vols)
    sim_mdds_arr = np.array(sim_mdds)
    
    # --- Plot 1: Return vs Volatility ---
    ret_min, ret_max = np.min(sim_returns_arr), np.max(sim_returns_arr)
    vol_min, vol_max = np.min(sim_vols_arr), np.max(sim_vols_arr)
    
    x_grid = np.linspace(ret_min, ret_max, 100)
    y_grid_vol = np.linspace(vol_min, vol_max, 100)
    X_vol, Y_vol = np.meshgrid(x_grid, y_grid_vol)
    Z_vol = np.zeros_like(X_vol)
    
    for i in range(X_vol.shape[0]):
        for j in range(X_vol.shape[1]):
            x = X_vol[i, j]
            y = Y_vol[i, j]
            prop = np.sum((sim_returns_arr > x) & (sim_vols_arr > y)) / n_simulations
            Z_vol[i, j] = prop
            
    vol_grids[N] = (X_vol, Y_vol, Z_vol)

    plt.figure(figsize=(10, 6))
    c1 = plt.contourf(X_vol, Y_vol, Z_vol, levels=50, cmap='viridis')
    plt.colorbar(c1, label='Proportion (Return > x & Vol > y)')
    
    cs1 = plt.contour(X_vol, Y_vol, Z_vol, levels=[0.05, 0.10, 0.25, 0.50], colors=['red', 'orange', 'yellow', 'white'], linewidths=2)
    plt.clabel(cs1, inline=True, fontsize=12, fmt='%.2f')

    plt.title(f'Monte Carlo Portfolios (N={N}) - Proportion(Ret > x & Vol > y)')
    plt.xlabel('Annualized Return (x)')
    plt.ylabel('Annualized Volatility (y)')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # --- Plot 2: Return vs MDD ---
    mdd_min, mdd_max = np.min(sim_mdds_arr), np.max(sim_mdds_arr)
    
    y_grid_mdd = np.linspace(mdd_min, mdd_max, 100)
    X_mdd, Y_mdd = np.meshgrid(x_grid, y_grid_mdd)
    Z_mdd = np.zeros_like(X_mdd)
    
    for i in range(X_mdd.shape[0]):
        for j in range(X_mdd.shape[1]):
            x = X_mdd[i, j]
            y = Y_mdd[i, j]
            prop = np.sum((sim_returns_arr > x) & (sim_mdds_arr > y)) / n_simulations
            Z_mdd[i, j] = prop
            
    mdd_grids[N] = (X_mdd, Y_mdd, Z_mdd)

    plt.figure(figsize=(10, 6))
    c2 = plt.contourf(X_mdd, Y_mdd, Z_mdd, levels=50, cmap='viridis')
    plt.colorbar(c2, label='Proportion (Return > x & MDD > y)')
    
    cs2 = plt.contour(X_mdd, Y_mdd, Z_mdd, levels=[0.05, 0.10, 0.25, 0.50], colors=['red', 'orange', 'yellow', 'white'], linewidths=2)
    plt.clabel(cs2, inline=True, fontsize=12, fmt='%.2f')

    plt.title(f'Monte Carlo Portfolios (N={N}) - Proportion(Ret > x & MDD > y)')
    plt.xlabel('Annualized Return (x)')
    plt.ylabel('Maximum Drawdown (y)')
    plt.grid(True, alpha=0.3)
    plt.show()

# --- Final Overlay Plots for Top 25% ---
if len(vol_grids) > 0:
    plt.figure(figsize=(10, 6))
    colors = {10: 'red', 15: 'orange', 20: 'blue'}
    
    for N, (X_vol, Y_vol, Z_vol) in vol_grids.items():
        cs = plt.contour(X_vol, Y_vol, Z_vol, levels=[0.25], colors=[colors[N]], linewidths=2)
        # dummy line for legend
        plt.plot([], [], color=colors[N], linewidth=2, label=f'N={N} (25%)')
        
    plt.title('Top 25% Frontier Comparison (Return vs Volatility)')
    plt.xlabel('Annualized Return (x)')
    plt.ylabel('Annualized Volatility (y)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

if len(mdd_grids) > 0:
    plt.figure(figsize=(10, 6))
    colors = {10: 'red', 15: 'orange', 20: 'blue'}
    
    for N, (X_mdd, Y_mdd, Z_mdd) in mdd_grids.items():
        cs = plt.contour(X_mdd, Y_mdd, Z_mdd, levels=[0.25], colors=[colors[N]], linewidths=2)
        plt.plot([], [], color=colors[N], linewidth=2, label=f'N={N} (25%)')
        
    plt.title('Top 25% Frontier Comparison (Return vs MDD)')
    plt.xlabel('Annualized Return (x)')
    plt.ylabel('Maximum Drawdown (y)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


In [ ]:
# Monte Carlo metrics vs N
def calculate_cagr(returns):
    cum_ret = (1 + returns).prod()
    n_years = len(returns) / 12
    return ((cum_ret ** (1 / n_years) - 1) * 100) if n_years > 0 else np.nan

def calculate_max_drawdown_duration(returns):
    cum_ret = (1 + returns).cumprod()
    running_max = cum_ret.cummax()
    is_high = (cum_ret == running_max)
    high_indices = np.where(is_high)[0]
    
    if len(high_indices) == 0:
        return len(returns)
        
    durations = np.diff(high_indices)
    last_duration = len(returns) - 1 - high_indices[-1]
    
    if len(durations) > 0:
        max_duration = max(durations.max(), last_duration)
    else:
        max_duration = last_duration
        
    return max_duration

N_values = [2, 4, 6, 8, 10, 15, 20, 25, 30]
n_sim_per_n = 1000

stats = {
    'mdd': {'N': [], 'median': [], 'q1': [], 'q3': []},
    'calmar': {'N': [], 'median': [], 'q1': [], 'q3': []},
    'vol': {'N': [], 'median': [], 'q1': [], 'q3': []},
    'sharpe': {'N': [], 'median': [], 'q1': [], 'q3': []},
    'duration': {'N': [], 'median': [], 'q1': [], 'q3': []},
    'cagr': {'N': [], 'median': [], 'q1': [], 'q3': []}
}

for n in tqdm(N_values, desc='Testing various N'):
    if n > len(available_funds):
        continue
    
    mdds, calmars, vols, sharpes, durations, cagrs = [], [], [], [], [], []
    for _ in range(n_sim_per_n):
        selected_funds = np.random.choice(available_funds, size=n, replace=False)
        portfolio_returns = test_data[selected_funds].mean(axis=1)
        
        # Calculate metrics
        mdd = calculate_mdd(portfolio_returns)
        cagr = calculate_cagr(portfolio_returns)
        calmar = cagr / abs(mdd) if mdd != 0 else np.nan
        
        vol = portfolio_returns.std() * np.sqrt(12)
        sharpe = (portfolio_returns.mean() / portfolio_returns.std()) * np.sqrt(12) if portfolio_returns.std() != 0 else np.nan
        duration = calculate_max_drawdown_duration(portfolio_returns)
        
        mdds.append(mdd)
        calmars.append(calmar)
        vols.append(vol)
        sharpes.append(sharpe)
        durations.append(duration)
        cagrs.append(cagr)
        
    for key, arr in zip(['mdd', 'calmar', 'vol', 'sharpe', 'duration', 'cagr'], [mdds, calmars, vols, sharpes, durations, cagrs]):
        stats[key]['N'].append(n)
        stats[key]['median'].append(np.nanmedian(arr))
        stats[key]['q1'].append(np.nanpercentile(arr, 25))
        stats[key]['q3'].append(np.nanpercentile(arr, 75))


In [ ]:
# Plotting Metrics vs N
if len(stats['mdd']['N']) > 0:
    metrics_info = [
        ('mdd', 'Max Drawdown vs Portfolio Size (N)', 'Max Drawdown', 'red'),
        ('calmar', 'Calmar Ratio vs Portfolio Size (N)', 'Calmar Ratio', 'blue'),
        ('vol', 'Annualized Volatility vs Portfolio Size (N)', 'Volatility', 'green'),
        ('sharpe', 'Sharpe Ratio vs Portfolio Size (N)', 'Sharpe Ratio', 'purple'),
        ('duration', 'Max Drawdown Duration (Months) vs Portfolio Size (N)', 'Duration (Months)', 'orange'),
        ('cagr', 'CAGR (%) vs Portfolio Size (N)', 'CAGR (%)', 'magenta')
    ]
    
    for key, title, ylabel, color in metrics_info:
        plt.figure(figsize=(10, 6))
        plt.plot(stats[key]['N'], stats[key]['median'], marker='o', label=f'Median {ylabel}', color=color)
        plt.fill_between(stats[key]['N'], stats[key]['q1'], stats[key]['q3'], color=color, alpha=0.2, label='Q1 - Q3 Range')
        plt.title(title)
        plt.xlabel('N (Number of Funds)')
        plt.ylabel(ylabel)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

else:
    print("Not enough funds for testing N_values.")
